In [ ]:
import sys, os, pickle, shutil
import numpy as np

_HERE         = os.path.abspath(os.getcwd())
_PROJECT_ROOT = os.path.dirname(_HERE)
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

from config import (
    CHUNKS_FOLDER,
    AGG_CHUNKS_FOLDER,
    AGGREGATED_GRAD_FILE_RAW,
)

def aggregate_raw_gradients():

    # Remove stale gradient file from previous round before producing a new one
    if os.path.exists(AGGREGATED_GRAD_FILE_RAW):
        os.remove(AGGREGATED_GRAD_FILE_RAW)
        print(f"[AGG] Removed stale {AGGREGATED_GRAD_FILE_RAW}")
    
    if not os.path.exists(CHUNKS_FOLDER):
        print(f"[AGG] No received_chunks_bin folder found at {CHUNKS_FOLDER}")
        return

    client_dirs = [
        d for d in os.listdir(CHUNKS_FOLDER)
        if os.path.isdir(os.path.join(CHUNKS_FOLDER, d))
    ]
    if not client_dirs:
        print("[AGG] No client gradients to aggregate.")
        return
    print(f"[AGG] Found {len(client_dirs)} client(s): {client_dirs}")

    chunk_groups = {}

    for client_id in client_dirs:
        client_path = os.path.join(CHUNKS_FOLDER, client_id)
        chunk_files = sorted(
            [f for f in os.listdir(client_path) if f.endswith(".bin")],
            key=lambda x: int(x.split("_")[-1].split(".")[0]),
        )
        for fname in chunk_files:
            try:
                idx = int(fname.split("_")[-1].split(".")[0])
                with open(os.path.join(client_path, fname), "rb") as f:
                    payload = pickle.load(f)
                if isinstance(payload, dict) and "data" in payload:
                    vec = np.array(payload["data"])
                    chunk_groups.setdefault(idx, []).append(vec)
                    print(f"[AGG] Loaded  {client_id}/{fname}")
                else:
                    print(f"[AGG] Invalid format in {fname} — skipped")
            except Exception as e:
                print(f"[AGG] Failed to load {fname}: {e}")

    if not chunk_groups:
        print("[AGG] No valid chunks to aggregate.")
        return

    aggregated_chunks = []
    for idx, vectors in sorted(chunk_groups.items()):
        try:
            stacked = np.stack(vectors)
            avg     = np.mean(stacked, axis=0)
            aggregated_chunks.append((idx, avg))
            print(f"[AGG] Aggregated chunk {idx}  "
                  f"({len(vectors)} clients, avg taken)")
        except Exception as e:
            print(f"[AGG] Failed to aggregate chunk {idx}: {e}")

    os.makedirs(AGG_CHUNKS_FOLDER, exist_ok=True)
    for idx, chunk in aggregated_chunks:
        path = os.path.join(AGG_CHUNKS_FOLDER, f"agg_chunk_{idx}.bin")
        with open(path, "wb") as f:
            pickle.dump(chunk, f)

    print(f"[AGG] {len(aggregated_chunks)} aggregated chunks saved "
          f"→ {AGG_CHUNKS_FOLDER}")

    all_arrays = [chunk for _, chunk in aggregated_chunks]
    with open(AGGREGATED_GRAD_FILE_RAW, "wb") as f:
        pickle.dump(all_arrays, f)
    print(f"[AGG] Global gradient written → {AGGREGATED_GRAD_FILE_RAW}")

    try:
        shutil.rmtree(CHUNKS_FOLDER)
        print(f"[AGG] Cleaned up {CHUNKS_FOLDER}")
    except Exception as e:
        print(f"[AGG] Cleanup failed: {e}")

aggregate_raw_gradients()
